In [35]:
import pandas as pd

In [36]:
df_a100 = pd.read_csv("../results/benchmark_bsrmv_A100.csv")
df_h100 = pd.read_csv("../results/benchmark_bsrmv_H100.csv")

In [37]:
df_a100["gpu"] = "A100"
df_h100["gpu"] = "H100"
df = pd.concat([df_a100, df_h100], ignore_index=True)

In [38]:
df["dtype"] = df["dtype"].str.replace("torch.float", "float")
df["elements"] = df.apply(
    lambda row: (
        2 * 4 ** row["mesh m"] if row["dimension"] == "2D" else 6 * 8 ** row["mesh m"]
    ),
    axis=1,
)

In [39]:
df.loc[
    (df["algorithm"] == "bsr_block_by_block") & (df["DOFs per element"] > 5),
    ["time [ms]"],
] = None

In [40]:
df

,dimension,mesh m,dtype,DOFs,DOFs per element,degree,algorithm,time [ms],error norm,gpu,elements
0,2D,11,float32,25165824,3,1,bsr_cusparse,4.694569,0.000000e+00,A100,8388608
1,2D,11,float32,25165824,3,1,bsr_row_per_thread_irrespective,1.029519,3.896300e-03,A100,8388608
2,2D,11,float32,25165824,3,1,bsr_column_by_column,6.539622,3.896300e-03,A100,8388608
3,2D,11,float32,25165824,3,1,bsr_block_by_block,6.527478,3.896300e-03,A100,8388608
4,2D,11,float32,25165824,3,1,bsr_row_per_thread,8.272394,3.896300e-03,A100,8388608
...,...,...,...,...,...,...,...,...,...,...,...
303,3D,5,float64,3932160,20,3,bsr_column_by_column,2.074609,8.459305e-14,H100,196608
304,3D,5,float64,3932160,20,3,bsr_block_by_block,NaN,8.459305e-14,H100,196608
305,3D,5,float64,3932160,20,3,bsr_row_per_thread,1.424208,8.459305e-14,H100,196608
306,3D,5,float64,3932160,20,3,bsr_row_per_thread_sync,1.402998,8.459305e-14,H100,196608


In [41]:
time_pivot = df.pivot_table(
    index=[
        "dimension",
        # "elements",
        "degree",
        "DOFs per element",
        "DOFs",
        "gpu",
        "dtype",
    ],
    columns=["algorithm"],
    values=["time [ms]"],
    aggfunc="min",
    sort=True,
)

In [42]:
ref_column = ("time [ms]", "csr_cusparse")
speedup_pivot = time_pivot.copy()
for col in time_pivot.columns:
    speedup_pivot[col] = time_pivot[ref_column] / time_pivot[col]
speedup_pivot.drop(columns=ref_column, inplace=True)

In [43]:
parameters_mapping = lambda x: {
    "dimension": "dim",
    "elements": "$N_h$",
    "degree": "$p$",
    "DOFs per element": "b",
    "gpu": "GPU",
}.get(x, x)

In [44]:
algorithm_mapping = {
    "block_by_block": "b-by-b",
    "column_by_column": "c-by-c",
    "cusparse": "cuSprs.",
    "row_per_thread_irrespective": "RPT-I",
    "row_per_thread_sync": "RPT-S",
    "row_per_thread": "RPT",
}

In [45]:
speedup_pivot.index.rename(
    list(map(parameters_mapping, speedup_pivot.index.names)), inplace=True
)

In [46]:
speedup_pivot.columns = speedup_pivot.columns.get_level_values(-1)

In [47]:
speedup_pivot = speedup_pivot[
    [
        "bsr_block_by_block",
        "bsr_column_by_column",
        "bsr_row_per_thread",
        "bsr_row_per_thread_sync",
        "bsr_row_per_thread_irrespective",
        "bsr_cusparse",
    ]
]

In [48]:
formatted_pivot = (
    speedup_pivot.rename(
        columns=lambda x: f"{algorithm_mapping[x.split('_', 1)[-1]]}" if "_" in x else x
    )
    .style.background_gradient(
        cmap="RdYlGn", vmin=-0.2, vmax=2.2, text_color_threshold=0
    )
    .map(lambda x: "background-color: white; color: black" if pd.isna(x) else "")
    .format(
        lambda x: (
            f"${x:.2f}\\times$"
            if isinstance(x, (float, int)) and not pd.isna(x)
            else "-"
        ),
    )
    .format_index(lambda x: f"{round(x / 1e5) / 10}\\,M", level="DOFs")
    .format_index(lambda x: f"fp{x.removeprefix('float')}", level="dtype")
    .hide(axis="columns", names=True)
)
formatted_pivot

In [49]:
index_columns = ["c"] * len(formatted_pivot.index.names)
value_columns = [r">{\centering\arraybackslash\hspace{0pt}}m{3.23em}"] * len(formatted_pivot.columns)
column_format = (
    "@{\\extracolsep{\\fill}}"
    + "@{\\hspace{7pt}}".join(index_columns)
    + "".join(value_columns)
)

pivot_latex = (
    formatted_pivot.to_latex(
        convert_css=True,
        column_format=column_format,
        hrules=True,
        # clines="skip-last;data",
        multirow_align="t",
    )
    .replace("\\begin{tabular}", "\\begin{tabular*}{\\textwidth}")
    .replace("\\end{tabular}", "\\end{tabular*}")
)

In [50]:
latex_lines = pivot_latex.split("\n")
first_header_line = latex_lines.index("\\toprule") + 1
last_header_line = latex_lines.index("\\midrule") - 1
column_names = [None] * len(latex_lines[first_header_line].split("&"))
for line in latex_lines[first_header_line : last_header_line + 1]:
    line_col_names = [x.strip().removesuffix("\\\\") for x in line.split("&")]
    for i, col_name in enumerate(line_col_names):
        if col_name:
            column_names[i] = col_name
single_header_line = " & ".join(column_names) + " \\\\"
hacked_pivot_latex = "\n".join(
    latex_lines[:first_header_line]
    + [single_header_line]
    + latex_lines[last_header_line + 1 :]
)

In [51]:
with open("../docs/tables/benchmark_bsrmv.tex", "w") as f:
    f.write(hacked_pivot_latex)